Setting up Drive and Dagshub/MLflow
---



In [1]:

from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/ML_final')

import torch
print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())

import pandas as pd
import numpy as np

Mounted at /content/drive
torch: 2.11.0+cu128 | CUDA: True


In [2]:
!pip install -q dagshub mlflow

import dagshub
import mlflow

from mlflow import MlflowClient
from mlflow.entities import ViewType

dagshub.init(
    repo_owner="skupr23",
    repo_name="ml_final",
    mlflow=True,
)

EXPERIMENT_NAME = "DLinear_Training"


if mlflow.active_run() is not None:
    mlflow.end_run()


client = MlflowClient()

# Include active and deleted experiments in the search
matches = client.search_experiments(
    view_type=ViewType.ALL,
    filter_string=f"name = '{EXPERIMENT_NAME}'",
)

if matches:
    experiment = matches[0]

    if experiment.lifecycle_stage == "deleted":
        client.restore_experiment(experiment.experiment_id)
        print(
            f"Restored deleted experiment: "
            f"{EXPERIMENT_NAME} ({experiment.experiment_id})"
        )

# Creates the experiment only if it genuinely does not exist
mlflow.set_experiment(EXPERIMENT_NAME)

active_experiment = client.get_experiment_by_name(EXPERIMENT_NAME)

print("Tracking URI :", mlflow.get_tracking_uri())
print("Experiment   :", active_experiment.name)
print("Experiment ID:", active_experiment.experiment_id)
print("Lifecycle    :", active_experiment.lifecycle_stage)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 131.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 130.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 89.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 131.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=d20a84f1-102b-4b64-89c9-ecd3d57d05cf&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=3fe2f99553b85e3b81cf0c5b65dfcf0558c422c1413ffe49c88ca6a6366440f2




Accessing as skupr23

Initialized MLflow to track repo "skupr23/ml_final"

Repository skupr23/ml_final initialized!

Tracking URI : https://dagshub.com/skupr23/ml_final.mlflow
Experiment   : DLinear_Training
Experiment ID: 9
Lifecycle    : active


# Cell 2 - Load processed data (identical source as the other notebooks)

In [3]:
train = pd.read_pickle('data/train_processed.pkl')
train = train.sort_values(['Store','Dept','Date'])
print(train['Date'].min(), train['Date'].max())

2010-02-05 00:00:00 2012-10-26 00:00:00


# Cell 3 - WMAE metric (same scoring function used everywhere)

In [4]:
def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5, 1)
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)

# Cell 4 - Rebuild the XGBoost / LightGBM holdout



In [5]:
VAL_WEEKS = 39

_t = train.sort_values('Date')
VAL_START = _t['Date'].max() - pd.Timedelta(weeks=VAL_WEEKS)
gbm_val = _t[_t['Date'] >= VAL_START].copy()
gbm_fit = _t[_t['Date'] < VAL_START].copy()

FIT_END = gbm_fit['Date'].max()
val_dates = pd.DatetimeIndex(np.sort(gbm_val['Date'].unique()))

print('fit ends:  ', FIT_END)
print('val range: ', gbm_val['Date'].min(), 'to', gbm_val['Date'].max())
print('val rows:  ', len(gbm_val),
      '| val dates:', len(val_dates),
      '| val pairs:', gbm_val[['Store','Dept']].drop_duplicates().shape[0])

baseline_pred = gbm_val['lag_52'].fillna(gbm_val['storedept_median_sales'])
baseline_wmae = wmae(gbm_val['Weekly_Sales'], baseline_pred, gbm_val['IsHoliday'])
print('\n52-week seasonal baseline WMAE:', baseline_wmae)
print('   XGBoost / LightGBM reported:  1789.9133525504185')
if abs(baseline_wmae - 1789.9133525504185) > 1e-6:
    raise AssertionError('population mismatch: this is not the GBM validation set')


sd_median = gbm_fit.groupby(['Store','Dept'])['Weekly_Sales'].median().to_dict()
d_median = gbm_fit.groupby('Dept')['Weekly_Sales'].median().to_dict()
g_median = float(gbm_fit['Weekly_Sales'].median())

fit ends:   2012-01-20 00:00:00
val range:  2012-01-27 00:00:00 to 2012-10-26 00:00:00
val rows:   118540 | val dates: 40 | val pairs: 3208

52-week seasonal baseline WMAE: 1789.9133525504185
   XGBoost / LightGBM reported:  1789.9133525504185


# Cell 5 - Per-series context arrays, built from the fit portion only



In [6]:
INPUT_LEN = 52
OUTPUT_LEN = 13

series_ctx = {}
n_ctx_interpolated = 0

for (store, dept), g in gbm_fit.groupby(['Store','Dept'], sort=False):
    if len(g) < INPUT_LEN:
        continue
    s = g.sort_values('Date').set_index('Date')['Weekly_Sales']

    full_idx = pd.date_range(s.index.min(), FIT_END, freq='W-FRI')
    s = s.reindex(full_idx)
    n_ctx_interpolated += int(s.isna().sum())
    s = s.interpolate().bfill().ffill()
    series_ctx[(store, dept)] = s.to_numpy(dtype=np.float32)

print('Series with a usable context window:', len(series_ctx))
print('Interpolated context points (model input only, never scored):', n_ctx_interpolated)

Series with a usable context window: 2940
Interpolated context points (model input only, never scored): 4574


# Cell 6 - Sliding (input, output) windows from the fit portion



In [7]:
X_train, Y_train, X_monitor, Y_monitor = [], [], [], []

for key, arr in series_ctx.items():
    n = len(arr)
    max_start = n - INPUT_LEN - OUTPUT_LEN
    if max_start < 1:
        continue
    for start in range(max_start):
        X_train.append(arr[start:start+INPUT_LEN])
        Y_train.append(arr[start+INPUT_LEN:start+INPUT_LEN+OUTPUT_LEN])
    X_monitor.append(arr[max_start:max_start+INPUT_LEN])
    Y_monitor.append(arr[max_start+INPUT_LEN:max_start+INPUT_LEN+OUTPUT_LEN])

X_train = np.stack(X_train).astype(np.float32)
Y_train = np.stack(Y_train).astype(np.float32)
X_monitor = np.stack(X_monitor).astype(np.float32)
Y_monitor = np.stack(Y_monitor).astype(np.float32)

print('Train windows:', X_train.shape, '| Monitor windows:', X_monitor.shape)

Train windows: (111366, 52) | Monitor windows: (2940, 52)


# Cell 7 - DLinear implementation in plain PyTorch



In [8]:
import torch.nn as nn

KERNEL_SIZE = 25


class RevIN(nn.Module):
    def __init__(self, eps=1e-5):
        super().__init__()
        self.eps = eps

    def forward(self, x):
        # x: (batch, input_len)
        mean = x.mean(dim=1, keepdim=True)
        std = x.std(dim=1, keepdim=True) + self.eps
        x_norm = (x - mean) / std
        return x_norm, mean, std


class MovingAvg(nn.Module):

    def __init__(self, kernel_size):
        super().__init__()
        assert kernel_size % 2 == 1, 'kernel_size must be odd to preserve length'
        self.kernel_size = kernel_size
        self.pad = (kernel_size - 1) // 2
        self.avg = nn.AvgPool1d(kernel_size=kernel_size, stride=1, padding=0)

    def forward(self, x):
        front = x[:, :1].repeat(1, self.pad)
        end = x[:, -1:].repeat(1, self.pad)
        padded = torch.cat([front, x, end], dim=1).unsqueeze(1)
        return self.avg(padded).squeeze(1)


class SeriesDecomp(nn.Module):

    def __init__(self, kernel_size):
        super().__init__()
        self.moving_avg = MovingAvg(kernel_size)

    def forward(self, x):
        trend = self.moving_avg(x)
        seasonal = x - trend
        return seasonal, trend


class DLinear(nn.Module):
    def __init__(self, input_len, output_len, kernel_size=KERNEL_SIZE):
        super().__init__()
        self.revin = RevIN()
        self.decomp = SeriesDecomp(kernel_size)
        self.linear_seasonal = nn.Linear(input_len, output_len)
        self.linear_trend = nn.Linear(input_len, output_len)

    def forward(self, x):
        x_norm, mean, std = self.revin(x)
        seasonal, trend = self.decomp(x_norm)
        out_norm = self.linear_seasonal(seasonal) + self.linear_trend(trend)
        out = out_norm * std + mean       # undo RevIN
        return out


device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_dlinear = DLinear(INPUT_LEN, OUTPUT_LEN).to(device)
print(model_dlinear)
print('Trainable parameters:', sum(p.numel() for p in model_dlinear.parameters()))

DLinear(
  (revin): RevIN()
  (decomp): SeriesDecomp(
    (moving_avg): MovingAvg(
      (avg): AvgPool1d(kernel_size=(25,), stride=(1,), padding=(0,))
    )
  )
  (linear_seasonal): Linear(in_features=52, out_features=13, bias=True)
  (linear_trend): Linear(in_features=52, out_features=13, bias=True)
)
Trainable parameters: 1378


# Cell 8 - Train with early stopping on the monitor set

In [9]:
from torch.utils.data import TensorDataset, DataLoader

X_train_t = torch.from_numpy(X_train)
Y_train_t = torch.from_numpy(Y_train)
X_monitor_t = torch.from_numpy(X_monitor).to(device)
Y_monitor_t = torch.from_numpy(Y_monitor).to(device)

train_loader = DataLoader(TensorDataset(X_train_t, Y_train_t), batch_size=1024, shuffle=True)

loss_fn = torch.nn.HuberLoss(delta=1.0)
optimizer = torch.optim.AdamW(model_dlinear.parameters(), lr=5e-4, weight_decay=1e-2)
N_EPOCHS = 30
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)

best_val_loss = float('inf')
patience, patience_counter = 5, 0
best_state = None
best_epoch = 0
history = []

for epoch in range(N_EPOCHS):
    model_dlinear.train()
    train_loss_sum, n_batches = 0.0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        pred = model_dlinear(xb)
        loss = loss_fn(pred, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_dlinear.parameters(), 0.5)
        optimizer.step()
        train_loss_sum += loss.item()
        n_batches += 1
    scheduler.step()

    model_dlinear.eval()
    with torch.no_grad():
        val_pred = model_dlinear(X_monitor_t)
        val_loss = loss_fn(val_pred, Y_monitor_t).item()

    train_loss = train_loss_sum / n_batches
    history.append({
        'epoch': epoch + 1,
        'train_loss': train_loss,
        'val_loss': val_loss,
        'lr': scheduler.get_last_lr()[0],
    })

    print(f"Epoch {epoch+1} - train_loss: {train_loss:.5f} - val_loss: {val_loss:.5f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.clone() for k, v in model_dlinear.state_dict().items()}
        best_epoch = epoch + 1
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early stopping.")
            break

epochs_trained = epoch + 1
model_dlinear.load_state_dict(best_state)

Epoch 1 - train_loss: 2693.27718 - val_loss: 3665.51343
Epoch 2 - train_loss: 2023.18812 - val_loss: 3305.32593
Epoch 3 - train_loss: 1815.86479 - val_loss: 3101.02612
Epoch 4 - train_loss: 1747.70459 - val_loss: 2949.89673
Epoch 5 - train_loss: 1714.49723 - val_loss: 2813.86963
Epoch 6 - train_loss: 1692.23564 - val_loss: 2709.01587
Epoch 7 - train_loss: 1675.77827 - val_loss: 2611.80127
Epoch 8 - train_loss: 1662.35858 - val_loss: 2529.68921
Epoch 9 - train_loss: 1651.54097 - val_loss: 2466.41357
Epoch 10 - train_loss: 1642.92120 - val_loss: 2402.64453
Epoch 11 - train_loss: 1636.13088 - val_loss: 2362.08032
Epoch 12 - train_loss: 1630.60021 - val_loss: 2314.84937
Epoch 13 - train_loss: 1626.50787 - val_loss: 2287.34668
Epoch 14 - train_loss: 1622.64360 - val_loss: 2257.69458
Epoch 15 - train_loss: 1620.06522 - val_loss: 2232.43628
Epoch 16 - train_loss: 1617.79922 - val_loss: 2214.69482
Epoch 17 - train_loss: 1615.77906 - val_loss: 2200.82300
Epoch 18 - train_loss: 1614.32931 - val_

<All keys matched successfully>

# Cell 9 - Forecast the 40 validation dates


In [10]:
N_BLOCKS = int(np.ceil(len(val_dates) / OUTPUT_LEN))
print('Rolling the', OUTPUT_LEN, 'week head forward', N_BLOCKS, 'times ->',
      N_BLOCKS * OUTPUT_LEN, 'weeks, truncated to', len(val_dates))

model_dlinear.eval()
gbm_val = gbm_val.sort_values(['Store','Dept','Date'])

pred_parts = []
n_model = n_fallback = 0

with torch.no_grad():
    for (store, dept), vg in gbm_val.groupby(['Store','Dept'], sort=False):
        arr = series_ctx.get((store, dept))

        if arr is None or len(arr) < INPUT_LEN:
            v = sd_median.get((store, dept), np.nan)
            if not np.isfinite(v):
                v = d_median.get(dept, np.nan)
            if not np.isfinite(v):
                v = g_median
            by_date = pd.Series(np.full(len(val_dates), float(v)), index=val_dates)
            n_fallback += 1
        else:
            context = arr[-INPUT_LEN:].copy()
            preds = []
            for _ in range(N_BLOCKS):
                x = torch.from_numpy(context[-INPUT_LEN:]).unsqueeze(0).to(device)
                block = model_dlinear(x).cpu().numpy().flatten()
                preds.extend(block)
                context = np.concatenate([context, block])
            by_date = pd.Series(np.asarray(preds[:len(val_dates)]), index=val_dates)
            n_model += 1

        pred_parts.append(pd.Series(by_date.reindex(vg['Date']).to_numpy(), index=vg.index))

all_pred_s = pd.concat(pred_parts)
scored = gbm_val.loc[all_pred_s.index]
all_pred = all_pred_s.to_numpy()
all_true = scored['Weekly_Sales'].to_numpy()
all_holiday = scored['IsHoliday'].to_numpy().astype(bool)

population_complete = len(scored) == len(gbm_val)
assert np.isfinite(all_pred).all(), 'a validation row did not receive a prediction'

print(f'\nDLinear forecasts: {n_model} pairs | median fallbacks: {n_fallback} pairs')
print('Rows scored:', len(scored), '| full population:', population_complete)

Rolling the 13 week head forward 4 times -> 52 weeks, truncated to 40

DLinear forecasts: 2934 pairs | median fallbacks: 274 pairs
Rows scored: 118540 | full population: True


# Cell 10 - Evaluate

In [11]:
dlinear_wmae = wmae(all_true, all_pred, all_holiday)
dlinear_wmae_holiday = wmae(all_true[all_holiday], all_pred[all_holiday], all_holiday[all_holiday])
dlinear_wmae_non_holiday = wmae(all_true[~all_holiday], all_pred[~all_holiday], all_holiday[~all_holiday])

_s = scored.assign(pred=all_pred)
_s['_w'] = np.where(all_holiday, 5, 1)
_s['_wae'] = _s['_w'] * (_s['Weekly_Sales'] - _s['pred']).abs()
per_pair = (
    _s.groupby(['Store','Dept'])
      .agg(_wae_sum=('_wae','sum'),
           _w_sum=('_w','sum'),
           mean_weekly_sales=('Weekly_Sales','mean'),
           n_val_weeks=('Weekly_Sales','size'))
      .reset_index()
)
per_pair['wmae'] = per_pair['_wae_sum'] / per_pair['_w_sum']
per_pair = (per_pair[['Store','Dept','wmae','mean_weekly_sales','n_val_weeks']]
            .sort_values('wmae')
            .reset_index(drop=True))

print('Baseline (52wk):  ', baseline_wmae)
print('DLinear WMAE:     ', dlinear_wmae)
print('  XGBoost:        ', 1281.080964551936)
print('  LightGBM:       ', 1297.4414789901796)
print('\nHoliday WMAE:    ', dlinear_wmae_holiday)
print('Non-holiday WMAE:', dlinear_wmae_non_holiday)

Baseline (52wk):   1789.9133525504185
DLinear WMAE:      1814.3220454017232
  XGBoost:         1281.080964551936
  LightGBM:        1297.4414789901796

Holiday WMAE:     2122.092788422725
Non-holiday WMAE: 1732.7541728387505


# Cell 11 - Save model locally (same pattern as the other notebooks)

In [12]:
import os
import joblib

os.makedirs('models', exist_ok=True)
joblib.dump(model_dlinear.cpu(), 'models/dlinear_best.pkl')
print('Saved.')

Saved.


# Cell 12 - MLflow Pipeline



In [13]:

import os
import tempfile

import cloudpickle
import mlflow.pytorch

PREPROCESSING_STEPS = [
    {'step': 'load_processed', 'detail': 'read data/train_processed.pkl, sort by Store-Dept-Date'},
    {'step': 'chronological_split',
     'detail': f"val = rows with Date >= max(Date) - {VAL_WEEKS} weeks; identical to the XGBoost/LightGBM notebooks"},
    {'step': 'no_series_filter_on_scoring',
     'detail': 'every Store-Dept pair in the validation window is scored; short pairs get a median fallback'},
    {'step': 'context_reindex',
     'detail': "asfreq('W-FRI') per series over the fit portion, ending at FIT_END; gaps interpolated"},
    {'step': 'interpolation_is_input_only',
     'detail': 'interpolated weeks are model input only; every scored row is a real observation'},
    {'step': 'windowing',
     'detail': f'sliding ({INPUT_LEN} -> {OUTPUT_LEN}) windows built from the fit portion only'},
    {'step': 'monitor_holdout',
     'detail': 'last fit window of each series reserved as the early-stopping monitor set'},
    {'step': 'instance_normalization', 'detail': 'RevIN inside the model; no global scaler is fitted'},
    {'step': 'fallback_aggregates',
     'detail': 'fit-portion median (store-dept -> dept -> global) for pairs with no context window'},
]


with mlflow.start_run(run_name='DLinear_Cleaning'):
    mlflow.set_tag('stage', 'cleaning')
    mlflow.log_params({
        'input_len': INPUT_LEN,
        'output_len': OUTPUT_LEN,
        'val_weeks': VAL_WEEKS,
        'resample_freq': 'W-FRI',
        'gap_fill': 'interpolate + bfill + ffill (context only)',
        'scaler': 'RevIN (per-window, inside the model)',
        'n_preprocessing_steps': len(PREPROCESSING_STEPS),
    })
    mlflow.log_metrics({
        'train_rows': train.shape[0],
        'train_cols': train.shape[1],
        'train_missing_cells': int(train.isna().sum().sum()),
        'n_stores': int(train['Store'].nunique()),
        'n_depts': int(train['Dept'].nunique()),
        'n_store_dept_pairs': int(train[['Store','Dept']].drop_duplicates().shape[0]),
        'fit_rows': len(gbm_fit),
        'val_rows': len(gbm_val),
        'val_dates': len(val_dates),
        'val_pairs': int(gbm_val[['Store','Dept']].drop_duplicates().shape[0]),
        'n_series_with_context': len(series_ctx),
        'n_context_interpolated_points': n_ctx_interpolated,
        'train_windows': X_train.shape[0],
        'monitor_windows': X_monitor.shape[0],
    })
    mlflow.log_dict({'steps': PREPROCESSING_STEPS}, 'preprocessing/pipeline.json')
    mlflow.log_dict({'columns': train.columns.tolist()}, 'preprocessing/processed_columns.json')
    mlflow.log_input(
        mlflow.data.from_pandas(train, name='train_processed', targets='Weekly_Sales'),
        context='training',
    )

with mlflow.start_run(run_name='DLinear_Training'):
    mlflow.set_tag('stage', 'training')
    mlflow.log_params({
        'architecture': 'DLinear (series decomposition + one linear head per component)',
        'kernel_size': KERNEL_SIZE,
        'normalization': 'RevIN (deviates from the paper, which standardizes globally)',
        'loss': 'HuberLoss(delta=1.0)',
        'optimizer': 'AdamW',
        'learning_rate': 5e-4,
        'weight_decay': 1e-2,
        'scheduler': 'CosineAnnealingLR',
        'batch_size': 1024,
        'grad_clip_norm': 0.5,
        'max_epochs': N_EPOCHS,
        'early_stopping_patience': patience,
        'device': device,
    })

    for h in history:
        mlflow.log_metrics(
            {'train_loss': h['train_loss'], 'val_loss': h['val_loss'], 'lr': h['lr']},
            step=h['epoch'],
        )

    mlflow.log_metrics({
        'best_val_loss': best_val_loss,
        'best_epoch': best_epoch,
        'epochs_trained': epochs_trained,
        'n_parameters': sum(p.numel() for p in model_dlinear.parameters()),
    })

    with tempfile.TemporaryDirectory() as tmp:
        p = os.path.join(tmp, 'training_history.csv')
        pd.DataFrame(history).to_csv(p, index=False)
        mlflow.log_artifact(p, artifact_path='training')

with mlflow.start_run(run_name='DLinear_Validation'):
    mlflow.set_tag('stage', 'validation')
    mlflow.set_tag('population', 'full' if population_complete else 'partial')
    mlflow.log_params({
        'scheme': 'chronological holdout (no k-fold: the target is a time series), rebuilt exactly as in the XGBoost/LightGBM notebooks',
        'val_weeks': VAL_WEEKS,
        'val_start': str(gbm_val['Date'].min().date()),
        'val_end': str(gbm_val['Date'].max().date()),
        'forecast_strategy': f'roll the {OUTPUT_LEN}-week head forward {N_BLOCKS}x, feeding predictions back as context, truncated to {len(val_dates)} weeks',
        'fallback': 'fit-portion median (store-dept -> dept -> global) for pairs with no context window',
        'population_complete': population_complete,
        'scored_on': 'same rows as XGBoost/LightGBM: directly comparable',
    })
    mlflow.log_metrics({
        'baseline_wmae': baseline_wmae,
        'final_wmae': dlinear_wmae,
        'val_wmae_holiday': dlinear_wmae_holiday,
        'val_wmae_non_holiday': dlinear_wmae_non_holiday,
        'improvement_over_baseline': baseline_wmae - dlinear_wmae,
        'val_rows': len(scored),
        'n_model_forecasts': n_model,
        'n_median_fallbacks': n_fallback,
        'per_pair_wmae_median': float(per_pair['wmae'].median()),

        'per_pair_wmae_vs_volume_corr': float(per_pair['wmae'].corr(per_pair['mean_weekly_sales'])),
    })
    with tempfile.TemporaryDirectory() as tmp:
        p = os.path.join(tmp, 'per_pair_wmae.csv')
        per_pair.to_csv(p, index=False)
        mlflow.log_artifact(p, artifact_path='validation')

with mlflow.start_run(run_name='DLinear_Final_Model') as final_run:
    mlflow.set_tags({'stage': 'final_model', 'algorithm': 'dlinear'})
    mlflow.log_params({
        'input_len': INPUT_LEN,
        'output_len': OUTPUT_LEN,
        'kernel_size': KERNEL_SIZE,
        'n_series_with_context': len(series_ctx),
        'best_epoch': best_epoch,
    })
    mlflow.log_metrics({
        'final_wmae': dlinear_wmae,
        'baseline_wmae': baseline_wmae,
        'best_val_loss': best_val_loss,
    })


    model_info = mlflow.pytorch.log_model(
        model_dlinear,
        artifact_path='model',
        pickle_module=cloudpickle,
        input_example=X_monitor[:5],
    )
    mlflow.log_artifact('models/dlinear_best.pkl', artifact_path='estimator')
    print('DLinear final run:', final_run.info.run_id)

/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


🏃 View run DLinear_Cleaning at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/9/runs/6d6c11396c3544b4a96be5347b24dcf8
🧪 View experiment at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/9
🏃 View run DLinear_Training at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/9/runs/9306369e22fb4b88a363972b91839099
🧪 View experiment at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/9
🏃 View run DLinear_Validation at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/9/runs/8df75becba444fe6a11d261eafc5503a
🧪 View experiment at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/9


2026/07/12 11:35:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/12 11:35:35 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/07/12 11:35:46 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmp71e0xqag/model/data, flavor: pytorch). Fall back to return ['torch==2.11.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 
/usr/local/lib/python3.12/dist-packages/torch/export/pt2_archive/_package.py:790: UserWarning: The given buffer is not writable, and PyTorch does not support non-writable tensors. This means you can write to the underlyin

DLinear final run: bc81816a4bde4c69a60a3d73c7486b1c
🏃 View run DLinear_Final_Model at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/9/runs/bc81816a4bde4c69a60a3d73c7486b1c
🧪 View experiment at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/9
